#  Text-to-Speech synthesis using Kokoro and OpenVINO

<div class="alert alert-block alert-danger"> <b>Important note:</b> This notebook requires python >= 3.10. Please make sure that your environment fulfill to this requirement before running it </div>

Kokoro is an TTS model with 82 million parameters. Despite its lightweight architecture, it delivers comparable quality to larger models while being significantly faster and more cost-efficient. More details about model can be found in [model card](https://huggingface.co/hexgrad/Kokoro-82M) and [original repository](https://github.com/hexgrad/kokoro)

In this tutorial, we consider how to run Kokoro using OpenVINO.
#### Table of contents:

- [Prerequisites](#Prerequisites)
- [Download and Run PyTorch model](#Download-and-Run-PyTorch-model)
- [Convert model to OpenVINO Intermediate Representation](#Convert-model-to-OpenVINO-Intermediate-Representation)
- [Run OpenVINO model](#Run-OpenVINO-model)
    - [Select inference device](#Select-inference-device)
- [Interactive Demo](#Interactive-Demo)


### Installation Instructions

This is a self-contained example that relies solely on its own code.

We recommend  running the notebook in a virtual environment. You only need a Jupyter server to start.
For details, please refer to [Installation Guide](https://github.com/openvinotoolkit/openvino_notebooks/blob/latest/README.md#-installation-guide).

For running this notebook, please make sure that `espeak-ng` installed:

**Linux**
```bash
apt-get -qq -y install espeak-ng > /dev/null 2>&1
```
**MacOS**
```bash
brew install espeak-ng
```
**Windows**
1. Go to [espeak-ng releases](https://github.com/espeak-ng/espeak-ng/releases)
2. Click on Latest release
3. Download the appropriate *.msi file (e.g. espeak-ng-20191129-b702b03-x64.msi
4. Run the downloaded installer

<img referrerpolicy="no-referrer-when-downgrade" src="https://static.scarf.sh/a.png?x-pxid=5b5a4db0-7875-4bfb-bdbd-01698b5b1a77&file=notebooks/kokoro/kokoro.ipynb" />

## Prerequisites
[back to top ⬆️](#Table-of-contents:)

In [ ]:
import platform
from pathlib import Path
import requests

%pip install -q "kokoro>=0.8.2" "misaki[en]>=0.8.2" soundfile "transformers!=4.49.0" --extra-index-url https://download.pytorch.org/whl/cpu
%pip install -q -U --pre "openvino>=2025.1.0" --extra-index-url https://storage.openvinotoolkit.org/simple/wheels/nightly

if platform.system() == "Darwin":
    %pip install -q "numpy<2.0"

if not Path("notebook_utils.py").exists():
    r = requests.get("https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/notebook_utils.py")
    with open("notebook_utils.py", "w") as f:
        f.write(r.text)

## Download and Run PyTorch model
[back to top ⬆️](#Table-of-contents:)

For running Kokoro TTS pipeline we should use `KPipeline` class. It will automatically download model from HuggingFace Hub. Additionally, we should provide language code for loading correct vocabulary. You can find list of supported languages and available voices for them in [model documentation](https://huggingface.co/hexgrad/Kokoro-82M/blob/main/VOICES.md)

In [1]:
# Read more about telemetry collection at https://github.com/openvinotoolkit/openvino_notebooks?tab=readme-ov-file#-telemetry
from notebook_utils import collect_telemetry

collect_telemetry("kokoro.ipynb")

from kokoro import KPipeline

model_id = "hexgrad/Kokoro-82M"

pipeline = KPipeline(lang_code="a", repo_id=model_id)

2025-04-17 18:53:10.813001: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-04-17 18:53:10.826856: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1744901590.841803 1399488 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1744901590.846081 1399488 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-04-17 18:53:10.862203: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

For running pipeline, we should provide input text for speech synthesis. Optionally, we can specify reference voice. The generation result contains generated audio, graphemes and phonemes for input text.

In [2]:
from IPython.display import Audio
import torch

input_text = "[Kokoro](/kˈOkəɹO/) is an open-weight TTS model with 82 million parameters. Despite its lightweight architecture, it delivers comparable quality to larger models while being significantly faster and more cost-efficient. With Apache-licensed weights, [Kokoro](/kˈOkəɹO/) can be deployed anywhere from production environments to personal projects."

with torch.no_grad():
    generator = pipeline(input_text, voice="af_heart")
    result = next(generator)

audio = Audio(data=result.audio, rate=24000)

display(audio)

## Convert model to OpenVINO Intermediate Representation
[back to top ⬆️](#Table-of-contents:)

Kokoro is PyTorch model. OpenVINO supports PyTorch models via conversion to OpenVINO Intermediate Representation (IR) format. You need to provide a model object, input data for model tracing to OpenVINO Model Conversion API. `ov.convert_model` function convert PyTorch model instance to `ov.Model` object that can be used for compilation on device or saved on disk using `ov.save_model` in compressed to FP16 format.

In [3]:
from pathlib import Path
import openvino as ov
from huggingface_hub import hf_hub_download
import gc

model_dir = Path(model_id.split("/")[-1])

if not (model_dir / "openvino_model.xml").exists():
    model = pipeline.model
    model.forward = model.forward_with_tokens
    input_ids = torch.randint(1, 100, (48,)).numpy()
    input_ids = torch.LongTensor([[0, *input_ids, 0]])
    style = torch.randn(1, 256)
    speed = torch.randint(1, 10, (1,), dtype=torch.float32)

    ov_model = ov.convert_model(model, example_input=(input_ids, style, speed), input=[ov.PartialShape("[1, 2..]"), ov.PartialShape([1, -1])])
    ov.save_model(ov_model, model_dir / "openvino_model.xml")
    hf_hub_download(repo_id=model_id, filename="config.json", local_dir=model_dir)

    del model
del pipeline

gc.collect()

`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.
/home/ea/work/py311/lib/python3.11/site-packages/kokoro/istftnet.py:380: TracerWarning: torch.tensor results are registered as constants in the trace. You can safely ignore this warning if you use this function to create tensors out of constant variables that would be the same every time you call this function. In any other case, this might cause the trace to be incorrect.
  out = (out + self._shortcut(x)) * torch.rsqrt(torch.tensor(2))
/home/ea/work/py311/lib/python3.11/site-packages/torch/jit/_trace.py:1304: TracerWarning: Output nr 1. of the traced function does not match the corresponding output of the Python function. Detailed error:
Tensor-likes are not close!

Mismatched elements: 29910 / 31200 (95.9%)
Greatest absolute difference: 1575691.65625 at index (28703,) (up to 1e-05 allowed)
Greatest relative difference: 81360.34839643467 at index (19672,) (up to 1e-05 allowed)
  _c

457

## Run OpenVINO model
[back to top ⬆️](#Table-of-contents:)

To be able use OpenVINO model with Kokoro pipeline, we provide model wrapper class that works like original Kokoro model.

In [7]:
import json
from kokoro.model import KModel

core = ov.Core()


class OVKModel(KModel):
    def __init__(self, model_dir: Path, device: str, repo_id: str = model_id):
        torch.nn.Module.__init__(self)
        self.repo_id = model_id
        with (model_dir / "config.json").open("r") as f:
            config = json.load(f)
        self.vocab = config["vocab"]
        self.model = core.compile_model(model_dir / "openvino_model.xml", device.upper())
        self.context_length = config["plbert"]["max_position_embeddings"]

    @property
    def device(self):
        return torch.device("cpu")

    def forward_with_tokens(self, input_ids: torch.LongTensor, ref_s: torch.FloatTensor, speed: float = 1) -> tuple[torch.FloatTensor, torch.LongTensor]:
        outputs = self.model([input_ids, ref_s, torch.tensor(speed)])
        return torch.from_numpy(outputs[0]), torch.from_numpy(outputs[1])

### Select inference device
[back to top ⬆️](#Table-of-contents:)

In [ ]:
from notebook_utils import device_widget

device = device_widget(default="CPU", exclude=["NPU"])

device

In [8]:
ov_model = OVKModel(model_dir, device.value)

ov_pipeline = KPipeline(lang_code="a", repo_id=model_id, model=ov_model)
generator = ov_pipeline(input_text, voice="af_heart")
result = next(generator)

audio = Audio(data=result.audio, rate=24000)

display(audio)

## Interactive Demo
[back to top ⬆️](#Table-of-contents:)

In [ ]:
if not Path("gradio_helper.py").exists():
    r = requests.get("https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/notebooks/kokoro/gradio_helper.py")
    with open("geadio_helper.py", "w") as f:
        f.write(r.text)

from gradio_helper import make_demo

demo = make_demo(ov_pipeline)

try:
    demo.launch(debug=True)
except Exception:
    demo.launch(share=True, debug=True)
# if you are launching remotely, specify server_name and server_port
# demo.launch(server_name='your server name', server_port='server port in int')
# Read more in the docs: https://gradio.app/docs/